# Assignment 01: Diabetes Classification System

## 1. System and Problem Definition

**Hệ thống: Phân loại Bệnh Tiểu đường**

- **Vấn đề:** Dự đoán nguy cơ mắc bệnh tiểu đường dựa trên các chỉ số sức khỏe.
- **Đầu vào:** `x` = [Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age]
- **Đầu ra:** `y` ∈ {0, 1} (0: Không tiểu đường, 1: Tiểu đường)
- **Loại bài toán:** Phân loại nhị phân (Binary Classification)

**Sơ đồ hệ thống:**
```
Chỉ số sức khỏe → [Tiền xử lý & Chuẩn hóa] → Vector đặc trưng → [ML Model] → Dự đoán (0/1) + Xác suất
```

## 2. Intelligent System Diagram

```
┌───────────────────────────────────────────────────────────────────────┐
│                    HỆ THỐNG PHÂN LOẠI TIỂU ĐƯỜNG                      │
└───────────────────────────────────────────────────────────────────────┘
                                 │
                                 ▼
┌───────────────────────────────────────────────────────────────────────┐
│  INPUT: Chỉ số sức khỏe của bệnh nhân                                 │
│  - Pregnancies (Số lần mang thai)                                     │
│  - Glucose (Nồng độ glucose)                                          │
│  - BloodPressure (Huyết áp)                                           │
│  - SkinThickness (Độ dày da)                                          │
│  - Insulin (Nồng độ insulin)                                          │
│  - BMI (Chỉ số khối cơ thể)                                           │
│  - DiabetesPedigreeFunction (Chỉ số di truyền)                        │
│  - Age (Tuổi)                                                         │
└───────────────────────────────────────────────────────────────────────┘
                                 │
                                 ▼
┌───────────────────────────────────────────────────────────────────────┐
│  REPRESENTATION: Feature Vector x ∈ R⁸                                │
│  - Tất cả đặc trưng đều là số                                         │
│  - Giá trị 0 bất thường → NaN → Điền bằng trung vị                    │
│  - Chuẩn hóa (StandardScaler): mean=0, std=1                          │
└───────────────────────────────────────────────────────────────────────┘
                                 │
                                 ▼
┌───────────────────────────────────────────────────────────────────────┐
│  MODEL: Random Forest (hoặc Logistic Regression, k-NN, Decision Tree, │
│          SVM)                                                         │
│  - Học mối quan hệ từ dữ liệu huấn luyện                              │
│  - Dự đoán nhãn và xác suất cho bệnh nhân mới                         │
└───────────────────────────────────────────────────────────────────────┘
                                 │
                                 ▼
┌───────────────────────────────────────────────────────────────────────┐
│  OUTPUT:                                                              │
│  - Có nguy cơ tiểu đường / Không có nguy cơ                           │
│  - Xác suất mắc bệnh (%)                                              │
│  - Mức độ rủi ro: Cao / Trung bình / Thấp                             │
└───────────────────────────────────────────────────────────────────────┘
```

## 3. Dataset Source

**Nguồn dữ liệu:** Pima Indians Diabetes Database (Kaggle)
- File: `diabetes.csv`
- Link: https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database

## 4. Dataset Description

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, classification_report)
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Đọc dữ liệu
diabetes_df = pd.read_csv(r"C:\DATA\diabetes.csv")
print("="*60)
print("THÔNG TIN DỮ LIỆU TIỂU ĐƯỜNG")
print("="*60)
print(f"Số lượng mẫu: {diabetes_df.shape[0]}")
print(f"Số lượng đặc trưng: {diabetes_df.shape[1] - 1}")  # trừ cột Outcome
print(f"\n5 mẫu đầu tiên:")
display(diabetes_df.head())

In [ ]:
print("\nTHÔNG TIN CÁC CỘT:")
print("-"*40)
print(diabetes_df.dtypes)
print("\nTHỐNG KÊ MÔ TẢ:")
display(diabetes_df.describe())

In [ ]:
print("\nPHÂN PHỐI LỚP (OUTCOME):")
print("-"*40)
print(diabetes_df['Outcome'].value_counts())
print(f"\nTỷ lệ: {diabetes_df['Outcome'].mean()*100:.2f}% bị tiểu đường")

## 5. Data Representation

**Biểu diễn dữ liệu:**

In [ ]:
# Tạo bảng biểu diễn đặc trưng
representation_df = pd.DataFrame({
    'Đặc trưng': ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 
                  'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
    'Kiểu dữ liệu': ['Numerical']*8 + ['Categorical'],
    'Biểu diễn': ['float']*8 + ['int'],
    'Ý nghĩa': ['Số lần mang thai', 'Nồng độ glucose', 'Huyết áp tâm trương', 
                'Độ dày nếp da', 'Mức insulin', 'Chỉ số khối cơ thể',
                'Chỉ số di truyền', 'Tuổi', 'Mục tiêu: 0 hoặc 1']
})
display(representation_df)

## 6. Feature and Target Analysis

In [ ]:
# Phân tích đặc trưng và mục tiêu
print("="*60)
print("PHÂN TÍCH ĐẶC TRƯNG VÀ MỤC TIÊU")
print("="*60)

# Thống kê chi tiết
display(diabetes_df.describe())

In [ ]:
# 3 biểu đồ phân phối
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Biểu đồ 1: Phân phối Glucose
sns.histplot(diabetes_df['Glucose'], bins=30, kde=True, ax=axes[0], color='blue')
axes[0].set_title('Phân phối Glucose', fontsize=14)
axes[0].set_xlabel('Glucose')
axes[0].set_ylabel('Tần suất')

# Biểu đồ 2: Phân phối BMI
sns.histplot(diabetes_df['BMI'], bins=30, kde=True, ax=axes[1], color='green')
axes[1].set_title('Phân phối BMI', fontsize=14)
axes[1].set_xlabel('BMI')
axes[1].set_ylabel('Tần suất')

# Biểu đồ 3: Phân phối Age
sns.histplot(diabetes_df['Age'], bins=30, kde=True, ax=axes[2], color='red')
axes[2].set_title('Phân phối Tuổi', fontsize=14)
axes[2].set_xlabel('Tuổi')
axes[2].set_ylabel('Tần suất')

plt.tight_layout()
plt.show()

In [ ]:
# Phân phối Outcome
plt.figure(figsize=(8, 5))
sns.countplot(x='Outcome', data=diabetes_df)
plt.title('Phân phối Outcome (0: Không tiểu đường, 1: Tiểu đường)', fontsize=14)
plt.xlabel('Outcome')
plt.ylabel('Số lượng')
for i, v in enumerate(diabetes_df['Outcome'].value_counts().sort_index()):
    plt.text(i, v + 10, str(v), ha='center', fontsize=12)
plt.show()

## 7. Exploratory Data Analysis (EDA)

In [ ]:
print("="*60)
print("PHÂN TÍCH KHÁM PHÁ DỮ LIỆU (EDA)")
print("="*60)

In [ ]:
# Tạo bản sao để xử lý giá trị 0
diabetes_clean = diabetes_df.copy()
cols_with_zero = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in cols_with_zero:
    diabetes_clean[col] = diabetes_clean[col].replace(0, np.nan)

print("Số lượng giá trị thiếu sau khi thay 0 bằng NaN:")
print(diabetes_clean.isnull().sum())

In [ ]:
# 3 biểu đồ phân phối cho EDA (so sánh giữa 2 lớp)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Biểu đồ 1: Glucose theo Outcome
sns.boxplot(x='Outcome', y='Glucose', data=diabetes_clean, ax=axes[0])
axes[0].set_title('Glucose theo Outcome', fontsize=14)

# Biểu đồ 2: BMI theo Outcome
sns.boxplot(x='Outcome', y='BMI', data=diabetes_clean, ax=axes[1])
axes[1].set_title('BMI theo Outcome', fontsize=14)

# Biểu đồ 3: Age theo Outcome
sns.boxplot(x='Outcome', y='Age', data=diabetes_clean, ax=axes[2])
axes[2].set_title('Tuổi theo Outcome', fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
# Ma trận tương quan
plt.figure(figsize=(10, 8))
correlation_matrix = diabetes_df.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', square=True)
plt.title('Ma trận tương quan các đặc trưng', fontsize=16)
plt.show()

## 8. Train/Test Split

In [ ]:
# Chuẩn bị dữ liệu
X_diabetes = diabetes_clean.drop('Outcome', axis=1)
y_diabetes = diabetes_clean['Outcome']

# Chia train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_diabetes, y_diabetes, test_size=0.20, random_state=42, stratify=y_diabetes
)

print("="*60)
print("CHIA DỮ LIỆU TRAIN/TEST")
print("="*60)
print(f"Train set: {X_train.shape[0]} mẫu")
print(f"Test set: {X_test.shape[0]} mẫu")
print(f"\nPhân phối lớp trong train set:")
print(y_train.value_counts())
print(f"\nPhân phối lớp trong test set:")
print(y_test.value_counts())

## 9. Baseline

In [ ]:
# Baseline: DummyClassifier (most_frequent)
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
y_pred_baseline = baseline.predict(X_test)

baseline_acc = accuracy_score(y_test, y_pred_baseline)
print("="*60)
print("BASELINE MODEL")
print("="*60)
print(f"Chiến lược: Dự đoán lớp phổ biến nhất")
print(f"Accuracy: {baseline_acc:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_baseline, target_names=['Không tiểu đường', 'Tiểu đường']))

## 10. Model 1: Logistic Regression

In [ ]:
# Model 1: Logistic Regression
print("="*60)
print("MODEL 1: LOGISTIC REGRESSION")
print("="*60)

# Pipeline với tiền xử lý và mô hình
lr_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42, C=1.0))
])

lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)
y_prob_lr = lr_pipeline.predict_proba(X_test)[:, 1]

# Đánh giá
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_lr):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_lr):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_lr):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_lr, target_names=['Không tiểu đường', 'Tiểu đường']))

# Confusion Matrix
cm_lr = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Logistic Regression')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## 11. Model 2: k-Nearest Neighbors

In [ ]:
# Model 2: k-NN
print("="*60)
print("MODEL 2: k-NEAREST NEIGHBORS (k=5)")
print("="*60)

knn_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('classifier', KNeighborsClassifier(n_neighbors=5))
])

knn_pipeline.fit(X_train, y_train)
y_pred_knn = knn_pipeline.predict(X_test)
y_prob_knn = knn_pipeline.predict_proba(X_test)[:, 1]

# Đánh giá
print(f"Accuracy: {accuracy_score(y_test, y_pred_knn):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_knn):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_knn):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_knn):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_knn, target_names=['Không tiểu đường', 'Tiểu đường']))

# Confusion Matrix
cm_knn = confusion_matrix(y_test, y_pred_knn)
sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - k-NN (k=5)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## 12. Model 3: Decision Tree

In [ ]:
# Model 3: Decision Tree
print("="*60)
print("MODEL 3: DECISION TREE")
print("="*60)

dt_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('classifier', DecisionTreeClassifier(random_state=42, max_depth=5))
])

dt_pipeline.fit(X_train, y_train)
y_pred_dt = dt_pipeline.predict(X_test)
y_prob_dt = dt_pipeline.predict_proba(X_test)[:, 1]

# Đánh giá
print(f"Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_dt):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_dt):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_dt):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_dt, target_names=['Không tiểu đường', 'Tiểu đường']))

# Confusion Matrix
cm_dt = confusion_matrix(y_test, y_pred_dt)
sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Decision Tree')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## 13. Model 4: Random Forest

In [ ]:
# Model 4: Random Forest
print("="*60)
print("MODEL 4: RANDOM FOREST")
print("="*60)

rf_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(random_state=42, n_estimators=100))
])

rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)
y_prob_rf = rf_pipeline.predict_proba(X_test)[:, 1]

# Đánh giá
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_rf):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_rf):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_rf):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_rf, target_names=['Không tiểu đường', 'Tiểu đường']))

# Confusion Matrix
cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Random Forest')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## 13.5 Model 5: Support Vector Machine (SVM)

In [ ]:
# Model 5: SVM
print("="*60)
print("MODEL 5: SUPPORT VECTOR MACHINE")
print("="*60)

svm_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('classifier', SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42))
])

svm_pipeline.fit(X_train, y_train)
y_pred_svm = svm_pipeline.predict(X_test)
y_prob_svm = svm_pipeline.predict_proba(X_test)[:, 1]

# Đánh giá
print(f"Accuracy: {accuracy_score(y_test, y_pred_svm):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_svm):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_svm):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_svm):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_svm, target_names=['Không tiểu đường', 'Tiểu đường']))

# Confusion Matrix
cm_svm = confusion_matrix(y_test, y_pred_svm)
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - SVM')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## 14. Evaluation

In [ ]:
# Tổng hợp kết quả
results_diabetes = {
    'Baseline': {'Accuracy': accuracy_score(y_test, y_pred_baseline),
                 'Precision': precision_score(y_test, y_pred_baseline, zero_division=0),
                 'Recall': recall_score(y_test, y_pred_baseline, zero_division=0),
                 'F1': f1_score(y_test, y_pred_baseline, zero_division=0)},
    'Logistic Regression': {'Accuracy': accuracy_score(y_test, y_pred_lr),
                            'Precision': precision_score(y_test, y_pred_lr),
                            'Recall': recall_score(y_test, y_pred_lr),
                            'F1': f1_score(y_test, y_pred_lr)},
    'k-NN (k=5)': {'Accuracy': accuracy_score(y_test, y_pred_knn),
                   'Precision': precision_score(y_test, y_pred_knn),
                   'Recall': recall_score(y_test, y_pred_knn),
                   'F1': f1_score(y_test, y_pred_knn)},
    'Decision Tree': {'Accuracy': accuracy_score(y_test, y_pred_dt),
                      'Precision': precision_score(y_test, y_pred_dt),
                      'Recall': recall_score(y_test, y_pred_dt),
                      'F1': f1_score(y_test, y_pred_dt)},
    'Random Forest': {'Accuracy': accuracy_score(y_test, y_pred_rf),
                      'Precision': precision_score(y_test, y_pred_rf),
                      'Recall': recall_score(y_test, y_pred_rf),
                      'F1': f1_score(y_test, y_pred_rf)},
    'SVM': {'Accuracy': accuracy_score(y_test, y_pred_svm),
            'Precision': precision_score(y_test, y_pred_svm),
            'Recall': recall_score(y_test, y_pred_svm),
            'F1': f1_score(y_test, y_pred_svm)}
}

# DataFrame
df_results = pd.DataFrame(results_diabetes).T
print("="*60)
print("TỔNG HỢP KẾT QUẢ CÁC MÔ HÌNH")
print("="*60)
display(df_results.round(4))

## 15. Experiment 1: Model Comparison

In [ ]:
# So sánh mô hình - Biểu đồ
fig, ax = plt.subplots(figsize=(12, 6))
df_results[['Accuracy', 'F1']].plot(kind='bar', ax=ax)
plt.title('So sánh các mô hình phân loại tiểu đường', fontsize=16)
plt.xlabel('Mô hình')
plt.ylabel('Điểm số')
plt.ylim(0, 1)
plt.xticks(rotation=45)
plt.legend(loc='lower right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 16. Experiment 2: Hyperparameter Investigation

### 16.1 k-NN: Thay đổi giá trị k

In [ ]:
print("="*60)
print("THÍ NGHIỆM 2A: ĐIỀU TRA SIÊU THAM SỐ k-NN")
print("="*60)

k_values = [3, 5, 7, 9, 11]
knn_results = {}

for k in k_values:
    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('classifier', KNeighborsClassifier(n_neighbors=k))
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    knn_results[k] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred)
    }

knn_df = pd.DataFrame(knn_results).T
display(knn_df.round(4))

# Biểu đồ
plt.figure(figsize=(8, 5))
plt.plot(knn_df.index, knn_df['Accuracy'], marker='o', label='Accuracy')
plt.plot(knn_df.index, knn_df['F1'], marker='s', label='F1-Score')
plt.xlabel('Số lượng láng giềng (k)')
plt.ylabel('Điểm số')
plt.title('Điều chỉnh siêu tham số k trong k-NN', fontsize=14)
plt.legend()
plt.grid(True)
plt.show()

### 16.2 Logistic Regression: Thay đổi C

In [ ]:
print("="*60)
print("THÍ NGHIỆM 2B: ĐIỀU TRA SIÊU THAM SỐ LOGISTIC REGRESSION")
print("="*60)

C_values = [0.01, 0.1, 1.0, 10.0, 100.0]
lr_results = {}

for C in C_values:
    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(max_iter=1000, random_state=42, C=C))
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    lr_results[C] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred)
    }

lr_df = pd.DataFrame(lr_results).T
display(lr_df.round(4))

# Biểu đồ
plt.figure(figsize=(8, 5))
plt.plot(lr_df.index, lr_df['Accuracy'], marker='o', label='Accuracy')
plt.plot(lr_df.index, lr_df['F1'], marker='s', label='F1-Score')
plt.xscale('log')
plt.xlabel('C (tham số điều chỉnh)')
plt.ylabel('Điểm số')
plt.title('Điều chỉnh siêu tham số C trong Logistic Regression', fontsize=14)
plt.legend()
plt.grid(True)
plt.show()

## 17. Experiment 3: Representation / Feature Investigation

In [ ]:
print("="*60)
print("THÍ NGHIỆM 3: ẢNH HƯỞNG CỦA CHUẨN HÓA (SVM)")
print("="*60)

In [ ]:
# Không chuẩn hóa
imputer = SimpleImputer(strategy='median')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

svm_no_scaling = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
svm_no_scaling.fit(X_train_imputed, y_train)
y_pred_no_scale = svm_no_scaling.predict(X_test_imputed)

acc_no_scale = accuracy_score(y_test, y_pred_no_scale)
f1_no_scale = f1_score(y_test, y_pred_no_scale)

# Có chuẩn hóa (lấy từ SVM pipeline)
svm_scaler = StandardScaler()
X_train_scaled = svm_scaler.fit_transform(X_train_imputed)
X_test_scaled = svm_scaler.transform(X_test_imputed)

svm_with_scaling = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
svm_with_scaling.fit(X_train_scaled, y_train)
y_pred_with_scale = svm_with_scaling.predict(X_test_scaled)

acc_with_scale = accuracy_score(y_test, y_pred_with_scale)
f1_with_scale = f1_score(y_test, y_pred_with_scale)

# So sánh
comparison_df = pd.DataFrame({
    'Phương pháp': ['Không chuẩn hóa', 'Có chuẩn hóa'],
    'Accuracy': [acc_no_scale, acc_with_scale],
    'F1-Score': [f1_no_scale, f1_with_scale]
})
display(comparison_df)

# Biểu đồ
fig, ax = plt.subplots(figsize=(8, 5))
comparison_df.set_index('Phương pháp')[['Accuracy', 'F1-Score']].plot(kind='bar', ax=ax)
plt.title('Ảnh hưởng của chuẩn hóa đến SVM', fontsize=14)
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.ylabel('Điểm số')
plt.legend(loc='lower right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 18. Final Model

In [ ]:
# Chọn mô hình tốt nhất
print("="*60)
print("MÔ HÌNH CUỐI CÙNG: RANDOM FOREST")
print("="*60)

# Huấn luyện lại với toàn bộ dữ liệu train
final_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(random_state=42, n_estimators=100))
])

final_pipeline.fit(X_train, y_train)

# Đánh giá trên test
y_pred_final = final_pipeline.predict(X_test)
y_prob_final = final_pipeline.predict_proba(X_test)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, y_pred_final):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_final):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_final):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_final):.4f}")

# Confusion Matrix
cm_final = confusion_matrix(y_test, y_pred_final)
sns.heatmap(cm_final, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Final Model (Random Forest)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## 19. Application

In [ ]:
# Lưu các thành phần cần thiết
diabetes_imputer = final_pipeline.named_steps['imputer']
diabetes_scaler = final_pipeline.named_steps['scaler']
diabetes_model = final_pipeline.named_steps['classifier']

def predict_diabetes(pregnancies, glucose, blood_pressure, skin_thickness,
                     insulin, bmi, diabetes_pedigree_function, age):
    """
    Dự đoán nguy cơ tiểu đường từ các chỉ số sức khỏe.
    """

    # 1. Tạo vector 8 đặc trưng
    raw_features = np.array([[
        pregnancies,
        glucose,
        blood_pressure,
        skin_thickness,
        insulin,
        bmi,
        diabetes_pedigree_function,
        age
    ]])

    # 2. Thay các giá trị 0 bất thường bằng NaN
    zero_cols = [1, 2, 3, 4, 5]

    for col in zero_cols:
        if raw_features[0, col] == 0:
            raw_features[0, col] = np.nan

    # 3. Điền giá trị thiếu
    imputed_features = diabetes_imputer.transform(raw_features)

    # 4. Chuẩn hóa
    scaled_features = diabetes_scaler.transform(imputed_features)

    # 5. Dự đoán
    prediction = diabetes_model.predict(scaled_features)
    probability = diabetes_model.predict_proba(scaled_features)[0][1]

    # 6. Xác định kết quả
    result = (
        "Có nguy cơ tiểu đường"
        if prediction[0] == 1
        else "Không có nguy cơ tiểu đường"
    )

    # 7. Phân loại mức độ rủi ro
    risk_level = (
        "Cao" if probability > 0.7
        else "Trung bình" if probability > 0.4
        else "Thấp"
    )

    return {
        'prediction': result,
        'probability': probability,
        'risk_level': risk_level
    }


## 20. System Demonstration

In [ ]:
print("="*60)
print("DEMO HỆ THỐNG PHÂN LOẠI TIỂU ĐƯỜNG")
print("="*60)

# Test cases: 8 đặc trưng
test_cases = [
    # Pregnancies, Glucose, BP, SkinThickness, Insulin,
    # BMI, DiabetesPedigreeFunction, Age
    (0, 148, 72, 35, 0, 33.6, 0.627, 50),
    (1, 85, 66, 29, 0, 26.6, 0.351, 31),
    (6, 148, 72, 35, 0, 33.6, 0.627, 50)
]

for i, case in enumerate(test_cases, 1):
    result = predict_diabetes(*case)

    print(f"\nCase {i}:")
    print(
        f"  Chỉ số: Pregnancies={case[0]}, "
        f"Glucose={case[1]}, "
        f"BMI={case[5]}, "
        f"DPF={case[6]}, "
        f"Age={case[7]}"
    )
    print(f"  → {result['prediction']}")
    print(f"  → Xác suất: {result['probability']:.2%}")
    print(f"  → Mức độ rủi ro: {result['risk_level']}")


In [ ]:
import joblib
import os

# Tạo thư mục models nếu chưa có
if not os.path.exists('models'):
    os.makedirs('models')

# Lưu toàn bộ pipeline (bao gồm imputer, scaler, model)
joblib.dump(final_pipeline, 'models/diabetes_model_pipeline.pkl')

# Hoặc lưu từng thành phần riêng lẻ
joblib.dump(diabetes_imputer, 'models/diabetes_imputer.pkl')
joblib.dump(diabetes_scaler, 'models/diabetes_scaler.pkl')
joblib.dump(diabetes_model, 'models/diabetes_model.pkl')

print("Đã lưu model thành công vào thư mục 'models/'")
print("- diabetes_model_pipeline.pkl (toàn bộ pipeline)")
print("- diabetes_imputer.pkl")
print("- diabetes_scaler.pkl")
print("- diabetes_model.pkl")

## 21. Reflection

**1. Hệ thống nhận thông tin gì?**
- Hệ thống nhận các chỉ số sức khỏe: Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age.

**2. Biểu diễn nội bộ là gì?**
- Biểu diễn dưới dạng vector số x ∈ R^8 sau khi được tiền xử lý (điền giá trị thiếu) và chuẩn hóa.

**3. Mô hình học gì từ các ví dụ?**
- Mô hình học mối quan hệ giữa vector đặc trưng và nhãn Outcome (0/1) từ dữ liệu huấn luyện.

**4. Dự đoán/Quyết định gì?**
- Dự đoán nhãn phân loại (0/1) và xác suất tương ứng.

**5. Tại sao có thể xử lý đầu vào chưa thấy?**
- Nhờ khả năng khái quát hóa của mô hình đã học từ dữ liệu huấn luyện.

**6. Phần nào có thể gọi là "thông minh"?**
- Phần mô hình tự động học và dự đoán là "thông minh".

**7. Hạn chế nào ngăn nó trở thành hệ thống thông minh mạnh mẽ hơn?**
- Dữ liệu nhỏ, thiếu các yếu tố quan trọng (lối sống, chế độ ăn), chưa học được biểu diễn phức tạp.

**Phân biệt trained model và complete intelligent system:**
- **Trained model:** Chỉ là hàm toán học Random Forest.
- **Complete intelligent system:** Bao gồm input handling, preprocessing, model, output generation, và ứng dụng demo.

## 22. Conclusion

**Kết luận:**

1. Xây dựng thành công hệ thống phân loại tiểu đường với 5 mô hình học máy.
2. Random Forest là mô hình tốt nhất (F1=0.681).
3. Chuẩn hóa dữ liệu cải thiện đáng kể hiệu suất (đặc biệt cho SVM).
4. Siêu tham số ảnh hưởng rõ rệt đến kết quả (k trong k-NN, C trong Logistic).
5. Hệ thống có thể dự đoán cho bệnh nhân mới với độ chính xác ~78%.
6. Cần cải thiện dữ liệu và biểu diễn để nâng cao hiệu suất.